# Mini ML Project — Wine Classification

**Goal:** predict which of three wine cultivars a sample belongs to,
based on 13 chemical measurements (alcohol content, acidity, color
intensity, etc.), and walk through the full ML workflow end to end.

**Dataset:** `../data/wine_dataset_W4D5.csv` (178 samples, 13 features,
3 classes) — the same data as scikit-learn's built-in wine dataset,
exported to CSV so this notebook reads it like any real-world dataset.

This notebook doubles as a reference for the full workflow covered this
week: load → explore → clean/preprocess → split → train → predict →
evaluate → compare models → conclude.

In [1]:
# --- Imports ---
# pandas/numpy for data handling, matplotlib/seaborn for charts,
# and the specific sklearn pieces for each pipeline stage
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
)

sns.set_theme(style="whitegrid")

## 1. Data Loading

Read the CSV from disk. `target` is the numeric class label (0, 1, 2);
`target_name` is the human-readable version we added when exporting the
dataset — we'll drop it before modeling since it's redundant with
`target`, but it's useful here for readable chart labels.

In [2]:
df = pd.read_csv("../data/wine_dataset_W4D5.csv")
print("Shape:", df.shape)
df.head()

Shape: (178, 15)


,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target,target_name
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0,class_0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0,class_0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0,class_0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0,class_0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0,class_0


## 2. Exploration

Before touching the model, look at what we're working with: are classes
balanced, are there missing values, and do the features look reasonable?

In [3]:
print("Class counts:\n", df["target_name"].value_counts())
print("\nMissing values:", df.isna().sum().sum())
df.describe().T[["mean", "std", "min", "max"]]

Class counts:
 target_name
class_1    71
class_0    59
class_2    48
Name: count, dtype: int64

Missing values: 0


,mean,std,min,max
alcohol,13.000618,0.811827,11.03,14.83
malic_acid,2.336348,1.117146,0.74,5.80
ash,2.366517,0.274344,1.36,3.23
alcalinity_of_ash,19.494944,3.339564,10.60,30.00
magnesium,99.741573,14.282484,70.00,162.00
total_phenols,2.295112,0.625851,0.98,3.88
flavanoids,2.029270,0.998859,0.34,5.08
nonflavanoid_phenols,0.361854,0.124453,0.13,0.66
proanthocyanins,1.590899,0.572359,0.41,3.58
color_intensity,5.058090,2.318286,1.28,13.00


In [4]:
# A quick visual: are classes separable on just two features?
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="alcohol", y="color_intensity", hue="target_name", palette="Set2")
plt.title("Alcohol vs. Color Intensity by Wine Class")
plt.tight_layout()
plt.savefig("../plots/feature_scatter_W4D5.png", dpi=150)
plt.show()

## 3. Preprocessing

Two things to handle before modeling:
1. Separate features (`X`) from the target (`y`), and drop the
   human-readable `target_name` column (it's not a feature — it's just
   `target` spelled out, and keeping it would leak the answer).
2. Scale the features with `StandardScaler`. This matters here because
   features like `proline` (hundreds) and `hue` (around 1) are on very
   different numeric scales — without scaling, KNN's distance
   calculations (and logistic regression's convergence) would be
   dominated by whichever feature happens to have the largest raw
   numbers, regardless of how predictive it actually is.

In [5]:
X = df.drop(columns=["target", "target_name"])
y = df["target"]

print("Feature columns:", list(X.columns))

Feature columns: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


## 4. Train-Test Split

We hold out 20% of the data as a test set the model never sees during
training, so the evaluation numbers reflect genuine generalization
rather than memorization. `stratify=y` keeps the same class proportions
in both the train and test sets, which matters for a multi-class
problem like this one. `random_state=42` just makes the split
reproducible — rerunning this notebook gives the same split every time.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

# Fit the scaler on TRAINING data only, then apply it to both sets.
# Fitting on the full dataset (including test data) would leak
# information about the test set into preprocessing - a common mistake.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train size: 142 | Test size: 36


## 5. Training

We'll train two different model types and compare them, rather than
committing to one blindly:
- **Logistic Regression** — assumes a roughly linear decision boundary
  between classes.
- **K-Nearest Neighbors (K=5)** — makes no assumption about the
  boundary's shape; classifies a point by majority vote among its
  5 closest neighbors in the scaled feature space.

In [7]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=5000),
    "KNN (K=5)": KNeighborsClassifier(n_neighbors=5),
}

# .fit() is where learning happens - the model finds the parameters
# (coefficients for logistic regression; just stores the training
# points for KNN) that best match X_train_scaled to y_train.
for model in models.values():
    model.fit(X_train_scaled, y_train)

print("Both models trained.")

Both models trained.


## 6. Prediction

`.predict()` runs each trained model on the held-out test features and
returns its class guess for each sample. These predictions have not
been compared to the true answers yet — that's the next step.

In [8]:
predictions = {name: model.predict(X_test_scaled) for name, model in models.items()}

print("First 10 predictions from each model (ground truth alongside):")
comparison_df = pd.DataFrame({
    "actual": y_test.values[:10],
    **{name: preds[:10] for name, preds in predictions.items()},
})
comparison_df

First 10 predictions from each model (ground truth alongside):


,actual,Logistic Regression,KNN (K=5)
0,0,0,0
1,2,1,2
2,0,0,0
3,1,1,1
4,1,1,1
5,0,0,0
6,0,0,0
7,1,1,1
8,1,1,1
9,2,2,2


## 7. Evaluation

Now we compare each model's predictions against the true test labels
using several metrics, since accuracy alone can hide weaknesses on
individual classes:
- **Accuracy** — overall fraction correct.
- **Precision/Recall/F1** (macro-averaged across the 3 classes) — catch
  cases where a model does well on one class but poorly on another.
- **Confusion matrix** — shows exactly which classes get confused with
  which.

In [9]:
results = {}
for name, preds in predictions.items():
    results[name] = {
        "accuracy": accuracy_score(y_test, preds),
        "precision": precision_score(y_test, preds, average="macro"),
        "recall": recall_score(y_test, preds, average="macro"),
        "f1": f1_score(y_test, preds, average="macro"),
    }

results_df = pd.DataFrame(results).T
results_df

,accuracy,precision,recall,f1
Logistic Regression,0.972222,0.977778,0.966667,0.970962
KNN (K=5),0.972222,0.969697,0.976190,0.971781


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["class_0", "class_1", "class_2"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(name)
fig.suptitle("Confusion Matrices")
fig.tight_layout()
fig.savefig("../plots/confusion_matrices_W4D5.png", dpi=150)
plt.show()

In [11]:
for name, preds in predictions.items():
    print(f"=== {name} ===")
    print(classification_report(y_test, preds, target_names=["class_0", "class_1", "class_2"]))

=== Logistic Regression ===
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       0.93      1.00      0.97        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36

=== KNN (K=5) ===
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        12
     class_1       1.00      0.93      0.96        14
     class_2       0.91      1.00      0.95        10

    accuracy                           0.97        36
   macro avg       0.97      0.98      0.97        36
weighted avg       0.97      0.97      0.97        36



## 8. Model Comparison and Conclusion

In [12]:
best_model = results_df["f1"].idxmax()
print(f"Best model by macro F1-score: {best_model}")
print(results_df.round(4))

Best model by macro F1-score: KNN (K=5)
                     accuracy  precision  recall      f1
Logistic Regression    0.9722     0.9778  0.9667  0.9710
KNN (K=5)              0.9722     0.9697  0.9762  0.9718


**Conclusion:** both models perform strongly on this dataset — it's a
clean, well-separated, moderately small dataset, which tends to favor
either a linear model or a distance-based one about equally. The small
performance gap (if any) between Logistic Regression and KNN here
matters less than the workflow itself: load, explore, preprocess with
no leakage, split before scaling, train multiple candidates, and
evaluate with more than one metric before declaring a winner. That
workflow — not any single model — is what generalizes to new datasets
and new problems.